LAB.CSV EDA
│
├── 1. Basic Analysis
│   ├── shape
│   ├── info
│   ├── dtypes
│   └── missing values
│
├── 2. Understand the columns
│   ├── ICU stay ID
│   ├── lab test ID/name
│   ├── time/offset
│   └── lab result/value
│
├── 3. ICU stay analysis
│   ├── unique ICU stays
│   ├── measurements per ICU stay
│   └── observation time range
│
├── 4. Numerical EDA
│   ├── distributions
│   ├── skewness
│   └── outliers
│
├── 5. Missing values
│
├── 6. Time analysis
│   └── observation offset
│
├── 7. Correlation
│
└── 8. EDA Findings

lab.csv — Complete EDA Plan

| Step | What we do                          | Status                        |
| ---- | ----------------------------------- | ----------------------------- |
| 1    | Shape, info, dtypes, missing values | ✅                             |
| 2    | Column classification               | ✅                             |
| 3    | ICU stay analysis                   | ✅                             |
| 4    | Lab-test analysis (`labname`)       | ✅                             |
| 5    | Numerical analysis of `labresult`   | ⬅️ Next                       |
| 6    | Distribution / skewness             | Later                         |
| 7    | Outliers                            | Later                         |
| 8    | Missing-value findings              | ✅ identified; treatment later |
| 9    | Time/offset analysis                | Later                         |
| 10   | Correlation                         | Later                         |
| 11   | Potential leakage                   | Later                         |
| 12   | Final EDA findings                  | Last                          |

after above

PREPROCESSING
  ↓
24-hour filtering + aggregation
  ↓
THEN perform the final 3-file common-ID checks

So yes, postpone the final common-ID/missing-ID/summary check until after aggregation.

## step 1: shape ,info,dtype,missingvalues

In [ ]:
import pandas as pd
import numpy as np

lab = pd.read_csv(r"C:\2)Projects\ICU_Patient_Predict_Risk_Analysis\dataset\lab.csv")

# shape

print("shape of file")
print(lab.shape)
print("-"*100)

#info
print("info of file")
print(lab.info())
print("-"*100)

# data types
print("datatypes of file")
print(lab.dtypes)
print("-"*100)


# mssing values
print("missing values of file")
print(lab.isnull().sum())


| Column                    | Type        | Purpose                                            |
| ------------------------- | ----------- | -------------------------------------------------- |
| `labid`                   | ID          | Unique lab record                                  |
| `patientunitstayid`       | ID          | Links lab record to ICU stay                       |
| `labresultoffset`         | Time        | When lab result occurred relative to ICU reference |
| `labtypeid`               | ID/category | Type of lab                                        |
| `labname`                 | Category    | Name of laboratory test                            |
| `labresult`               | Numerical   | Numeric lab result                                 |
| `labresulttext`           | Text        | Result represented as text                         |
| `labmeasurenamesystem`    | Category    | Measurement name from system                       |
| `labmeasurenameinterface` | Category    | Measurement name from interface                    |
| `labresultrevisedoffset`  | Time        | Time of result revision                            |


## Step2 : column classification

In [ ]:


# ID columns
lab_id_cols = [
    "labid",
    "patientunitstayid",
    "labtypeid"
]

# Time columns
lab_time_cols = [
    "labresultoffset",
    "labresultrevisedoffset"
]

# Numerical measurement
lab_numeric_cols = [
    "labresult"
]

# Categorical columns
lab_categorical_cols = [
    "labname",
    "labmeasurenamesystem",
    "labmeasurenameinterface"
]

# Text/result representation
lab_text_cols = [
    "labresulttext"
]

Important for our ML project

The most important columns will eventually be:

patientunitstayid
        ↓
connect lab → ICU patient

labresultoffset
        ↓
select first 24 hours

labname
        ↓
identify which test

labresult
        ↓
actual numerical value

Step 3 — Understand ICU stays and number of lab measurements

Just like vitalperiodic.csv, we need to understand the repeated structure.

In [ ]:
print("Number of unique ICU stays:")
print(lab["patientunitstayid"].nunique())

print("-" * 80)

print("Number of lab measurements per ICU stay:")
print(lab.groupby("patientunitstayid").size().describe())

print("-" * 80)

print("Lab observation time:")
print(lab["labresultoffset"].describe())

| Finding                   |     Your result | Meaning                                           |
| ------------------------- | --------------: | ------------------------------------------------- |
| Total lab records         |     **434,660** | Total individual lab-result rows                  |
| Unique ICU stays          |       **2,444** | Labs belong to 2,444 ICU stays                    |
| Average measurements/stay |      **177.85** | Each ICU stay has ~178 lab records on average     |
| Minimum/stay              |           **1** | Some stays have only 1 lab record                 |
| Median/stay               |         **121** | Half of ICU stays have ≤121 records               |
| Maximum/stay              |       **2,727** | One ICU stay has 2,727 lab records                |
| Minimum offset            | **-68,394 min** | Some records occur before the ICU reference point |
| Maximum offset            |  **90,072 min** | Some records occur much later                     |
| Median offset             |   **1,993 min** | ~33.2 hours                                       |


Step 4 — Find which laboratory tests are present  ( LAB TEST Analysis)

In [ ]:
print("Number of unique lab tests: \n ")
print(lab["labname"].nunique())
print("-"*200)


print("Number of value_counts: \n ")
lab_test_counts = lab["labname"].value_counts()
print(lab_test_counts)
print("-"*200)


print("Top 20 lab tests:")
print(lab["labname"].value_counts().head(20))
print("-"*200)


print("Missing lab names:")
print(lab["labname"].isnull().sum())

patient.csv
   │
   │ patient information
   │
   ├── patientunitstayid
   │
   ▼
vitalperiodic.csv
   │
   │ repeated vital measurements
   │
   ├── patientunitstayid
   │
   ▼
lab.csv
      │
      │ repeated laboratory measurements
      │
      └── patientunitstayid


| File                |                  ICU stays |
| ------------------- | -------------------------: |
| `patient.csv`       |      2,520 rows originally |
| `vitalperiodic.csv` | **2,375** unique ICU stays |
| `lab.csv`           | **2,444** unique ICU stays |

These numbers are not necessarily supposed to be equal.

Some ICU stays may have:

patient information but no VitalPeriodic records
patient information but no lab records
patient + vitals + labs

For our final ML dataset, the last category is especially important.

Stpe5: labresult - Numerical analysis 


In [ ]:
#IMP NOTE
 
'''
 Q) Why used labname?

The key idea

labname tells us what was measured:

labname          labresult
--------------------------
glucose             120
glucose             135
potassium             4.2
potassium             4.5
sodium               140

So we use:

groupby("labname")["labresult"]

to ask:

"What is the distribution of glucose values?"
"What is the distribution of potassium values?"
"What is the distribution of sodium values?"

We cannot combine all lab columns together because they represent different things.

'''

In [ ]:
# 5.1 Basic numerical statistics

# First, check the overall numeric labresult availability:
print("Total lab records:", len(lab))

print("Numeric lab results:", lab["labresult"].notna().sum())

print("Missing lab results:", lab["labresult"].isna().sum())

In [ ]:
# 5.2 Number of numerical results for each lab test

lab_numeric_counts = (
    lab.groupby("labname")["labresult"].count()
    .sort_values(ascending=False)
)


print("total values",len(lab_numeric_counts))
print("*" *150)

print(lab_numeric_counts.head(20))

5.3 Select important lab tests

In [ ]:
#  We have 147 tests, but some tests occur only once or twice.

# For meaningful EDA, initially focus on tests with enough observations.

# this prevents the one-off tests from producing meaningless statistics.

important_lab_tests = lab_numeric_counts[
    lab_numeric_counts >= 100
].index

print("Number of lab tests with at least 100 results:",len(important_lab_tests))
print()


In [ ]:
# 5.4 statistics for each important lab tests

lab_stats = (
    lab[lab["labname"].isin(important_lab_tests)]
    .groupby("labname")["labresult"]
    .describe()
)

lab_stats

In [ ]:
'''

This helps identify values that may need investigation later.
Because different lab tests have completely different units, we must check ranges separately.

This helps identify values that may need investigation later.

Important: Don't automatically delete them.

For example, an extreme value might be:

true clinical value

or:

data-entry/unit problem

We decide that during data cleaning/preprocessing, not EDA.


'''

# 5.5 Find potentially Unusual ranges
lab_ranges = (
    lab[lab["labname"].isin(important_lab_tests)]
    .groupby("labname")["labresult"]
    .agg(["min", "max"])
    .sort_values("max", ascending=False)
)

lab_ranges

5.6 Check zero values

Some laboratory tests may legitimately contain zero, while others may not.

In [ ]:
zero_counts = (
    lab.groupby("labname")["labresult"]
    .apply(lambda x: (x == 0).sum())
    .sort_values(ascending=False)
)

print(zero_counts.head(20))

#5.7 Check negative values

#This is also important because some lab measurements should normally not be negative.

In [ ]:
negative_counts = (
    lab.groupby("labname")["labresult"]
    .apply(lambda x: (x < 0).sum())
    .sort_values(ascending=False)
)

print(negative_counts.head(20))

In [ ]:
# 5.8 Find which tests have suspicious negative values
negative_tests = negative_counts[negative_counts > 0]

print(negative_tests)

In [ ]:
# 5.9 Find potential outliers using IQR

# For lab data, outliers must also be calculated per lab test.

outlier_results = []

for test in important_lab_tests:

    values = lab.loc[lab["labname"] == test, "labresult"].dropna()

    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((values < lower) | (values > upper)).sum()

    outlier_results.append([
        test,
        len(values),
        Q1,
        Q3,
        lower,
        upper,
        outliers
    ])

lab_outliers = pd.DataFrame(
    outlier_results,
    columns=[
        "Lab_Test",
        "Count",
        "Q1",
        "Q3",
        "Lower_Bound",
        "Upper_Bound",
        "Outlier_Count"
    ]
)

print(lab_outliers.head(10))

lab_outliers = lab_outliers.sort_values(
    "Outlier_Count",
    ascending=False
)

print("-"*80)
lab_outliers.head(20)

In [ ]:
# 5.10 Skewness per lab test

# After that, calculate skewness:

lab_skewness = (
    lab[lab["labname"].isin(important_lab_tests)]
    .groupby("labname")["labresult"]
    .skew()
    .sort_values()
)

print(lab_skewness)

In [ ]:
# 5.11 Distribution plots

# We should not create 147 histograms.

# Start with the most common 10 tests

top_10_lab_tests = lab_numeric_counts.head(10).index

for test in top_10_lab_tests:

    values = lab.loc[
        lab["labname"] == test, "labresult"
    ].dropna()

    values.plot(
        kind="hist",
        bins=30,
        title=f"Distribution of {test}"
    )

    import matplotlib.pyplot as plt
    plt.xlabel("Lab Result")
    plt.show()

In [ ]:
# STEP 9: TIME Analysis


#labresultoffset

In [ ]:
print("Minimum offset:", lab["labresultoffset"].min())
print("-"*135)
print("Maximum offset:", lab["labresultoffset"].max())
print("-"*135)


# Then count negative and non-negative values:

print("Negative:", (lab["labresultoffset"] < 0).sum())
print("-"*135)

print("Zero or positive:", (lab["labresultoffset"] >= 0).sum())
print("-"*135)
print("Missing:", lab["labresultoffset"].isna().sum())
print("-"*135)


# Convert minutes to hours so it's easier to understand:
lab["labresult_hours"] = lab["labresultoffset"] / 60

print(lab["labresult_hours"].describe())

In [ ]:

#Step 8 — Correlation

'''
Here we should correlate numerical lab features, but there is an important difference from vitalperiodic.csv.

You currently have only:

labresult

as the actual clinical numerical value.

labresultoffset is time, and IDs should not be treated as clinical features.

So do not calculate a giant correlation matrix across IDs and categorical columns.

For lab.csv, correlation becomes meaningful after we transform the repeated lab records into 
patient/ICU-stay-level features.


For example, later:

ICU stay 1001

glucose_mean
glucose_max
sodium_mean
potassium_mean
creatinine_max
...

Then we can calculate correlations between these engineered numerical features.

Therefore for EDA now:

You can document:

Correlation between individual lab tests cannot be directly calculated
from the raw labresult column because different lab tests represent
different measurements and units. Correlation analysis will be performed
after lab results are transformed into ICU-stay-level numerical features
during preprocessing.

This is the correct approach.


'''